In [ ]:
import json
import tempfile
from pathlib import Path

import mlflow
import numpy as np
from PIL import Image, ImageDraw
import matplotlib.pyplot as plt

base_dir = Path("Files/mlflow_demo")
base_dir.mkdir(parents=True, exist_ok=True)

# Files/mlflow_demo 配下に一時ディレクトリを作成
temp_dir = Path(tempfile.mkdtemp(prefix="sample_files_", dir=base_dir))

single_file_dir = temp_dir / "single_file"
batch_dir = temp_dir / "batch_dir"
image_dir = temp_dir / "local_images"

single_file_dir.mkdir(parents=True, exist_ok=True)
batch_dir.mkdir(parents=True, exist_ok=True)
image_dir.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# 1. log_artifact 用の単一ファイルを生成
# ------------------------------------------------------------
sample_txt_path = single_file_dir / "sample.txt"
sample_txt_path.write_text(
    "これは log_artifact 用のサンプルファイルです。\n"
    "1行目\n"
    "2行目\n",
    encoding="utf-8"
)

# ------------------------------------------------------------
# 2. log_artifacts 用の複数ファイルを生成
# ------------------------------------------------------------
(batch_dir / "report1.txt").write_text("report 1\n", encoding="utf-8")
(batch_dir / "report2.txt").write_text("report 2\n", encoding="utf-8")

nested_dir = batch_dir / "nested"
nested_dir.mkdir(exist_ok=True)

with open(nested_dir / "meta.json", "w", encoding="utf-8") as f:
    json.dump({"source": "sample", "version": 1}, f, ensure_ascii=False, indent=2)

# ------------------------------------------------------------
# 3. log_image 用の画像を生成
# ------------------------------------------------------------
# NumPy 画像
img_array = np.full((128, 128, 3), 240, dtype=np.uint8)
img_array[32:96, 32:96] = [255, 0, 0]

# Pillow 画像
pil_img = Image.new("RGB", (128, 128), "white")
draw = ImageDraw.Draw(pil_img)
draw.rectangle((16, 16, 112, 112), outline="blue", width=4)
draw.ellipse((40, 40, 88, 88), fill="green")

# 参考用にローカル保存もしておく
local_pil_img_path = image_dir / "pillow_sample_local.png"
pil_img.save(local_pil_img_path)

# ------------------------------------------------------------
# 4. log_figure 用の matplotlib figure を生成
# ------------------------------------------------------------
x = np.arange(10)
y = x ** 2

fig, ax = plt.subplots()
ax.plot(x, y, marker="o")
ax.set_title("y = x^2")
ax.set_xlabel("x")
ax.set_ylabel("y")

# ------------------------------------------------------------
# 5. メトリクス時系列サンプル
#    同じ key に対して別の値を step ごとに記録する
# ------------------------------------------------------------
epochs = [0, 1, 2, 3, 4]
train_losses = [0.92, 0.71, 0.55, 0.41, 0.33]
val_losses = [0.95, 0.76, 0.60, 0.48, 0.39]
val_accs = [0.72, 0.79, 0.84, 0.88, 0.91]

# ------------------------------------------------------------
# 6. MLflow ロギング関数を試験
# ------------------------------------------------------------
with mlflow.start_run(run_name="all_mlflow_logging_samples"):
    # ------------------------
    # log_param
    # ------------------------
    mlflow.log_param("optimizer", "adam")

    # ------------------------
    # log_params
    # ------------------------
    mlflow.log_params({
        "learning_rate": 0.001,
        "batch_size": 32,
        "epochs": len(epochs),
    })

    # ------------------------
    # log_metric
    # 同じ key で別の値を連続記録
    # ------------------------
    for epoch, train_loss, val_loss, val_acc in zip(epochs, train_losses, val_losses, val_accs):
        mlflow.log_metric("train_loss", train_loss, step=epoch)
        mlflow.log_metric("val_loss", val_loss, step=epoch)
        mlflow.log_metric("val_accuracy", val_acc, step=epoch)

    # 単発 metric
    mlflow.log_metric("best_val_accuracy", max(val_accs))

    # ------------------------
    # log_metrics
    # ------------------------
    mlflow.log_metrics({
        "final_train_loss": train_losses[-1],
        "final_val_loss": val_losses[-1],
        "final_val_accuracy": val_accs[-1],
    })

    # ------------------------
    # log_artifact
    # ------------------------
    mlflow.log_artifact(str(sample_txt_path), artifact_path="single_files")

    # ------------------------
    # log_artifacts
    # ------------------------
    mlflow.log_artifacts(str(batch_dir), artifact_path="batch_files")

    # ------------------------
    # log_text
    # ------------------------
    mlflow.log_text("1行目\n2行目\n3行目", "texts/example.txt")
    mlflow.log_text(
        "<h1>MLflow HTML sample</h1><p>hello</p>",
        "texts/example.html"
    )

    # ------------------------
    # log_dict
    # ------------------------
    config = {
        "model": "xgboost",
        "learning_rate": 0.1,
        "features": ["age", "income", "score"],
        "enabled": True,
    }
    mlflow.log_dict(config, "configs/config.json")
    mlflow.log_dict(config, "configs/config.yaml")

    # ------------------------
    # log_image
    # ------------------------
    # 同じ key で step を変えて記録する例
    mlflow.log_image(img_array, key="training_samples/sample_image", step=0)

    # artifact ファイルとして保存する例
    mlflow.log_image(pil_img, artifact_file="images/pillow_sample.png")

    # ------------------------
    # log_figure
    # ------------------------
    mlflow.log_figure(fig, artifact_file="figures/quadratic_plot.png")

    active_run = mlflow.active_run()
    if active_run is not None:
        print("run_id:", active_run.info.run_id)

plt.close(fig)